<a href="https://colab.research.google.com/github/gabrieldaim/QuantVisionBench/blob/main/treino_yolo_dataset_degradado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Instalar dependências
!pip install ultralytics -q
!pip install opencv-python -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.4 MB/s eta 0:00:00


In [ ]:
#Importações
import os
import zipfile
import shutil
from pathlib import Path
from PIL import Image

In [ ]:
#Descompactar os zips
BASE_DIR = Path("/content")
ZIP_TRAIN = BASE_DIR / "VisDrone2019-DET-train.zip"
ZIP_VAL = BASE_DIR / "VisDrone2019-DET-val.zip"

RAW_DIR = BASE_DIR / "visdrone_raw"

RAW_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_TRAIN, 'r') as zip_ref:
    zip_ref.extractall(RAW_DIR)

with zipfile.ZipFile(ZIP_VAL, 'r') as zip_ref:
    zip_ref.extractall(RAW_DIR)

print("Arquivos extraídos com sucesso!")

Arquivos extraídos com sucesso!


In [ ]:
#Verificar estrutura
!find /content/visdrone_raw -maxdepth 3 -type d
#Você deve ver algo parecido com:
#VisDrone2019-DET-train/images
#VisDrone2019-DET-train/annotations
#VisDrone2019-DET-val/images
#VisDrone2019-DET-val/annotations

/content/visdrone_raw
/content/visdrone_raw/VisDrone2019-DET-val
/content/visdrone_raw/VisDrone2019-DET-val/images
/content/visdrone_raw/VisDrone2019-DET-val/annotations
/content/visdrone_raw/VisDrone2019-DET-train
/content/visdrone_raw/VisDrone2019-DET-train/images
/content/visdrone_raw/VisDrone2019-DET-train/annotations


In [ ]:
#Criar estrutura YOLO
YOLO_DIR = BASE_DIR / "visdrone_yolo"

for split in ["train", "val"]:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Estrutura YOLO criada.")

Estrutura YOLO criada.


In [ ]:
#Conversor VisDrone → YOLO
#No VisDrone, as classes vão de 1 a 10.
#No YOLO, precisam ir de 0 a 9.
def convert_visdrone_to_yolo(image_dir, annotation_dir, output_image_dir, output_label_dir):
    image_dir = Path(image_dir)
    annotation_dir = Path(annotation_dir)
    output_image_dir = Path(output_image_dir)
    output_label_dir = Path(output_label_dir)

    image_files = list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.png"))

    print(f"Encontradas {len(image_files)} imagens em {image_dir}")

    for image_path in image_files:
        image = Image.open(image_path)
        img_w, img_h = image.size

        # Copia imagem
        shutil.copy(image_path, output_image_dir / image_path.name)

        annotation_path = annotation_dir / f"{image_path.stem}.txt"
        output_label_path = output_label_dir / f"{image_path.stem}.txt"

        yolo_lines = []

        if annotation_path.exists():
            with open(annotation_path, "r") as f:
                lines = f.readlines()

            for line in lines:
                values = line.strip().split(",")

                if len(values) < 8:
                    continue

                x, y, w, h = map(float, values[:4])
                score = int(values[4])
                category = int(values[5])

                # Ignora regiões marcadas como não avaliáveis
                if score == 0:
                    continue

                # Categoria 0 no VisDrone é ignored region
                if category == 0:
                    continue

                # VisDrone: 1 a 10
                # YOLO: 0 a 9
                class_id = category - 1

                x_center = (x + w / 2) / img_w
                y_center = (y + h / 2) / img_h
                w_norm = w / img_w
                h_norm = h / img_h

                # Evita valores inválidos
                if w_norm <= 0 or h_norm <= 0:
                    continue

                yolo_lines.append(
                    f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
                )

        with open(output_label_path, "w") as f:
            f.write("\n".join(yolo_lines))

In [ ]:
#Converter treino e validação
TRAIN_RAW = RAW_DIR / "VisDrone2019-DET-train"
VAL_RAW = RAW_DIR / "VisDrone2019-DET-val"

convert_visdrone_to_yolo(
    image_dir=TRAIN_RAW / "images",
    annotation_dir=TRAIN_RAW / "annotations",
    output_image_dir=YOLO_DIR / "images" / "train",
    output_label_dir=YOLO_DIR / "labels" / "train"
)

convert_visdrone_to_yolo(
    image_dir=VAL_RAW / "images",
    annotation_dir=VAL_RAW / "annotations",
    output_image_dir=YOLO_DIR / "images" / "val",
    output_label_dir=YOLO_DIR / "labels" / "val"
)

print("Conversão finalizada!")

Encontradas 6471 imagens em /content/visdrone_raw/VisDrone2019-DET-train/images
Encontradas 548 imagens em /content/visdrone_raw/VisDrone2019-DET-val/images
Conversão finalizada!


In [ ]:
#Criar o YAML do dataset
yaml_content = f"""
path: {YOLO_DIR}
train: images/train
val: images/val

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

yaml_path = BASE_DIR / "visdrone.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(yaml_path)
print(yaml_content)

/content/visdrone.yaml

path: /content/visdrone_yolo
train: images/train
val: images/val

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor



In [1]:
#Treinar YOLO26 Small
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data="/content/visdrone.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    patience=20,
    cache=True,
    workers=2,
    name="yolo26n_visdrone_fp32"
)

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/visdrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_visdrone_fp32, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

KeyboardInterrupt: 

In [ ]:
#Validar modelo FP32
model_fp32 = YOLO("/content/runs/detect/yolo11n_visdrone_fp32/weights/best.pt")

metrics_fp32 = model_fp32.val(
    data="/content/visdrone.yaml",
    imgsz=640,
    device=0
)

print(metrics_fp32)

In [2]:
#Exportar para INT8
model_fp32 = YOLO("/content/runs/detect/yolo26n_visdrone_fp32/weights/best.pt")

model_fp32.export(
    format="openvino",
    int8=True,
    data="/content/visdrone.yaml",
    imgsz=640
)

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,376,786 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/content/runs/detect/yolo26n_visdrone_fp32/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (14.8 MB)
requirements: Ultralytics requirement ['openvino>=2024.0.0'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 3 packages in 349ms
Prepared 2 packages in 3.92s
Installed 2 packages in 28ms
 + openvino==2026.2.0
 + openvino-telemetry==2025.2.0

requirements: AutoUpdate success ✅ 4.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirement ['nncf>=2.14.0'] not found, attempting AutoUpdate...
Using Python 3.12.13 envi

Output()

Output()

OpenVINO: export success ✅ 101.1s, saved as '/content/runs/detect/yolo26n_visdrone_fp32/weights/best_int8_openvino_model/' (3.2 MB)

Export complete (101.6s)
Results saved to /content/runs/detect/yolo26n_visdrone_fp32/weights/best_int8_openvino_model
Predict:         yolo predict task=detect model=/content/runs/detect/yolo26n_visdrone_fp32/weights/best_int8_openvino_model/ imgsz=640 int8
Validate:        yolo val task=detect model=/content/runs/detect/yolo26n_visdrone_fp32/weights/best_int8_openvino_model/ imgsz=640 data=/content/visdrone.yaml int8 
Visualize:       https://netron.app


'/content/runs/detect/yolo26n_visdrone_fp32/weights/best_int8_openvino_model/'

In [ ]:
#Para usar

from ultralytics import YOLO

model_fp32 = YOLO("weights/best.pt")

model_int8 = YOLO("weights/best_int8_openvino_model/")

model_fp32.predict("imagem.jpg")
model_int8.predict("imagem.jpg")

In [3]:
#baixar a pasta do modelo

!zip -r best_int8_openvino_model.zip /content/runs/detect/yolo26n_visdrone_fp32

from google.colab import files

files.download("best_int8_openvino_model.zip")


  adding: content/runs/detect/yolo26n_visdrone_fp32/ (stored 0%)
  adding: content/runs/detect/yolo26n_visdrone_fp32/weights/ (stored 0%)
  adding: content/runs/detect/yolo26n_visdrone_fp32/weights/best_int8_openvino_model/ (stored 0%)
  adding: content/runs/detect/yolo26n_visdrone_fp32/weights/best_int8_openvino_model/metadata.yaml (deflated 41%)
  adding: content/runs/detect/yolo26n_visdrone_fp32/weights/best_int8_openvino_model/best.xml (deflated 95%)
  adding: content/runs/detect/yolo26n_visdrone_fp32/weights/best_int8_openvino_model/best.bin (deflated 16%)
  adding: content/runs/detect/yolo26n_visdrone_fp32/weights/last.pt (deflated 10%)
  adding: content/runs/detect/yolo26n_visdrone_fp32/weights/best.pt (deflated 10%)
  adding: content/runs/detect/yolo26n_visdrone_fp32/train_batch2.jpg (deflated 2%)
  adding: content/runs/detect/yolo26n_visdrone_fp32/train_batch1.jpg (deflated 3%)
  adding: content/runs/detect/yolo26n_visdrone_fp32/args.yaml (deflated 53%)
  adding: content/runs/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>